In [ ]:
# SECTION 9: Summary and Export

# Summary statistics
summary = {
    "extraction_timestamp": datetime.utcnow().isoformat(),
    "collections_count": len(all_collections),
    "data_sources_count": len(all_sources),
    "assets_count": len(all_assets),
    "scans_count": len(all_scans),
    "classifications_count": len(all_classifications),
    "runtimes_count": len(all_runtimes),
    "accounts_processed": [acc.get("name") if isinstance(acc, dict) else acc for acc in all_accounts],
    "primary_account": primary_account
}

print("\n" + "="*60)
print("PURVIEW EXTRACTION SUMMARY")
print("="*60)
for key, value in summary.items():
    print(f"{key}: {value}")

# Store as global variables for next notebook
try:
    purview_extraction_data = {
        "collections": all_collections,
        "sources": all_sources,
        "assets": all_assets,
        "scans": all_scans,
        "classifications": all_classifications,
        "runtimes": all_runtimes,
        "summary": summary
    }
    
    # Save to shared location or notebook widget
    print("\n✓ Extraction data ready for next notebook (02-UnifiedCatalog-Extract)")
except Exception as e:
    logger.error(f"Error storing extraction data: {e}")


In [ ]:
# SECTION 8: Extract Integration Runtimes

def extract_runtimes(purview_client: PurviewClient, account_name: str) -> List[Dict]:
    """Extract all Integration Runtime configurations"""
    
    logger.info(f"Extracting Integration Runtimes from {account_name}")
    
    try:
        runtimes = purview_client.get_runtimes()
        
        runtime_list = []
        for runtime_id, runtime_data in runtimes.items():
            record = {
                "runtime_id": runtime_id,
                "runtime_name": runtime_data.get("name"),
                "runtime_type": runtime_data.get("type"),
                "status": runtime_data.get("status"),
                "description": runtime_data.get("description"),
                "environment": runtime_data.get("environment"),
                "source_account": account_name,
                "is_primary": account_name == primary_account,
                "extracted_at": datetime.utcnow().isoformat(),
                "raw_json": json.dumps(runtime_data)
            }
            runtime_list.append(record)
        
        logger.info(f"✓ Extracted {len(runtime_list)} runtimes")
        return runtime_list
    
    except Exception as e:
        logger.error(f"Error extracting runtimes: {e}")
        return []

# Extract from all accounts
all_runtimes = []
for account in all_accounts:
    account_name = account.get("name") if isinstance(account, dict) else account
    try:
        client = PurviewClient(credential)
        client.set_account(f"https://{account_name}.purview.azure.com")
        
        runtimes = extract_runtimes(client, account_name)
        all_runtimes.extend(runtimes)
    except Exception as e:
        logger.error(f"Cannot access account {account_name}: {e}")

print(f"✓ Total runtimes extracted: {len(all_runtimes)}")

if all_runtimes:
    runtime_df = spark.createDataFrame(all_runtimes)
    runtime_df.display()


In [ ]:
# SECTION 7: Extract Classifications and Glossary

def extract_classifications(purview_client: PurviewClient, account_name: str) -> List[Dict]:
    """Extract all classifications and glossary terms"""
    
    logger.info(f"Extracting Classifications from {account_name}")
    
    try:
        classifications = purview_client.get_classifications()
        
        class_list = []
        for class_id, class_data in classifications.items():
            record = {
                "classification_id": class_id,
                "classification_name": class_data.get("name"),
                "display_name": class_data.get("displayName"),
                "category": class_data.get("category"),
                "description": class_data.get("description"),
                "usage_count": class_data.get("usageCount", 0),
                "source_account": account_name,
                "is_primary": account_name == primary_account,
                "extracted_at": datetime.utcnow().isoformat(),
                "raw_json": json.dumps(class_data)
            }
            class_list.append(record)
        
        logger.info(f"✓ Extracted {len(class_list)} classifications")
        return class_list
    
    except Exception as e:
        logger.error(f"Error extracting classifications: {e}")
        return []

# Extract from all accounts
all_classifications = []
for account in all_accounts:
    account_name = account.get("name") if isinstance(account, dict) else account
    try:
        client = PurviewClient(credential)
        client.set_account(f"https://{account_name}.purview.azure.com")
        
        classifications = extract_classifications(client, account_name)
        all_classifications.extend(classifications)
    except Exception as e:
        logger.error(f"Cannot access account {account_name}: {e}")

print(f"✓ Total classifications extracted: {len(all_classifications)}")

if all_classifications:
    class_df = spark.createDataFrame(all_classifications)
    class_df.display()


In [ ]:
# SECTION 6: Extract Scans and Scan Runs

def extract_scans(purview_client: PurviewClient, account_name: str) -> List[Dict]:
    """Extract all scan configurations"""
    
    logger.info(f"Extracting Scans from {account_name}")
    
    try:
        scans = purview_client.get_scans()
        
        scans_list = []
        for scan_id, scan_data in scans.items():
            record = {
                "scan_id": scan_id,
                "scan_name": scan_data.get("name"),
                "scan_rule_set": scan_data.get("scanRulesetName"),
                "data_source_id": scan_data.get("dataSourceId"),
                "scan_status": scan_data.get("status"),
                "last_run": scan_data.get("lastScannedOn"),
                "source_account": account_name,
                "is_primary": account_name == primary_account,
                "extracted_at": datetime.utcnow().isoformat(),
                "raw_json": json.dumps(scan_data)
            }
            scans_list.append(record)
        
        logger.info(f"✓ Extracted {len(scans_list)} scans")
        return scans_list
    
    except Exception as e:
        logger.error(f"Error extracting scans: {e}")
        return []

# Extract from all accounts
all_scans = []
for account in all_accounts:
    account_name = account.get("name") if isinstance(account, dict) else account
    try:
        client = PurviewClient(credential)
        client.set_account(f"https://{account_name}.purview.azure.com")
        
        scans = extract_scans(client, account_name)
        all_scans.extend(scans)
    except Exception as e:
        logger.error(f"Cannot access account {account_name}: {e}")

print(f"✓ Total scans extracted: {len(all_scans)}")

if all_scans:
    scans_df = spark.createDataFrame(all_scans)
    scans_df.display()


In [ ]:
# SECTION 5: Extract Assets and Lineage

def extract_assets(purview_client: PurviewClient, account_name: str, limit: int = None) -> List[Dict]:
    """Extract all cataloged assets with basic properties"""
    
    logger.info(f"Extracting Assets from {account_name}")
    
    try:
        assets = purview_client.get_assets()
        
        assets_list = []
        for asset_id, asset_data in assets.items():
            if limit and len(assets_list) >= limit:
                break
            
            record = {
                "asset_id": asset_id,
                "asset_name": asset_data.get("name"),
                "asset_type": asset_data.get("typeName"),
                "collection_id": asset_data.get("collectionId"),
                "classifications": ",".join([c.get("typeName") for c in asset_data.get("classifications", [])]),
                "has_lineage": len(asset_data.get("lineage", [])) > 0,
                "owner": asset_data.get("owner"),
                "source_account": account_name,
                "is_primary": account_name == primary_account,
                "extracted_at": datetime.utcnow().isoformat(),
                "raw_json": json.dumps(asset_data)
            }
            assets_list.append(record)
        
        logger.info(f"✓ Extracted {len(assets_list)} assets")
        return assets_list
    
    except Exception as e:
        logger.error(f"Error extracting assets: {e}")
        return []

# Extract from all accounts (with reasonable limit for demo)
all_assets = []
for account in all_accounts:
    account_name = account.get("name") if isinstance(account, dict) else account
    try:
        client = PurviewClient(credential)
        client.set_account(f"https://{account_name}.purview.azure.com")
        
        # Limit assets per account for memory efficiency
        assets = extract_assets(client, account_name, limit=1000)
        all_assets.extend(assets)
    except Exception as e:
        logger.error(f"Cannot access account {account_name}: {e}")

print(f"✓ Total assets extracted: {len(all_assets)}")

if all_assets:
    assets_df = spark.createDataFrame(all_assets)
    print(f"Asset type distribution:")
    assets_df.groupby("asset_type").count().show(20)


In [ ]:
# SECTION 4: Extract Data Sources

def extract_data_sources(purview_client: PurviewClient, account_name: str) -> List[Dict]:
    """Extract all data sources and their scan configurations"""
    
    logger.info(f"Extracting Data Sources from {account_name}")
    
    try:
        sources = purview_client.get_data_sources()
        
        sources_list = []
        for source_id, source_data in sources.items():
            record = {
                "source_id": source_id,
                "source_name": source_data.get("name"),
                "source_type": source_data.get("sourceType"),
                "endpoint": source_data.get("endpoint"),
                "collection_id": source_data.get("collectionId"),
                "has_scans": len(source_data.get("scans", [])) > 0,
                "scan_count": len(source_data.get("scans", [])),
                "source_account": account_name,
                "is_primary": account_name == primary_account,
                "extracted_at": datetime.utcnow().isoformat(),
                "raw_json": json.dumps(source_data)
            }
            sources_list.append(record)
        
        logger.info(f"✓ Extracted {len(sources_list)} data sources")
        return sources_list
    
    except Exception as e:
        logger.error(f"Error extracting data sources: {e}")
        return []

# Extract from all accounts
all_sources = []
for account in all_accounts:
    account_name = account.get("name") if isinstance(account, dict) else account
    try:
        client = PurviewClient(credential)
        client.set_account(f"https://{account_name}.purview.azure.com")
        
        sources = extract_data_sources(client, account_name)
        all_sources.extend(sources)
    except Exception as e:
        logger.error(f"Cannot access account {account_name}: {e}")

print(f"✓ Total data sources extracted: {len(all_sources)}")

if all_sources:
    sources_df = spark.createDataFrame(all_sources)
    sources_df.display()


In [ ]:
# SECTION 3: Extract Collections Hierarchy

def extract_collections(purview_client: PurviewClient, account_name: str) -> Dict[str, Any]:
    """Extract all collections and their hierarchy"""
    
    logger.info(f"Extracting Collections from {account_name}")
    
    try:
        collections = purview_client.get_collections()
        
        # Normalize to DataFrame schema
        collections_list = []
        for coll_id, coll_data in collections.items():
            record = {
                "collection_id": coll_id,
                "collection_name": coll_data.get("name"),
                "parent_collection_id": coll_data.get("parentCollectionId"),
                "friendly_name": coll_data.get("friendlyName"),
                "description": coll_data.get("description"),
                "source_account": account_name,
                "is_primary": account_name == primary_account,
                "extracted_at": datetime.utcnow().isoformat(),
                "raw_json": json.dumps(coll_data)
            }
            collections_list.append(record)
        
        logger.info(f"✓ Extracted {len(collections_list)} collections")
        return collections_list
    
    except Exception as e:
        logger.error(f"Error extracting collections: {e}")
        return []

# Extract from all accounts
all_collections = []
for account in all_accounts:
    account_name = account.get("name") if isinstance(account, dict) else account
    try:
        client = PurviewClient(credential)
        client.set_account(f"https://{account_name}.purview.azure.com")
        
        collections = extract_collections(client, account_name)
        all_collections.extend(collections)
    except Exception as e:
        logger.error(f"Cannot access account {account_name}: {e}")

print(f"✓ Total collections extracted: {len(all_collections)}")

# Create DataFrame
if all_collections:
    collections_df = spark.createDataFrame(all_collections)
    collections_df.display()


In [ ]:
# SECTION 2: Retrieve Audit Configuration from Previous Notebook

# These variables should be passed from 00-Setup or stored in shared location
# In Databricks/Synapse, you can use notebook parameters or DButils

try:
    # Variables passed from Setup notebook
    primary_account = spark.sql("SELECT spark_master_account FROM default.audit_config LIMIT 1").collect()[0][0]
    all_accounts = json.loads(spark.sql("SELECT purview_accounts FROM default.audit_config LIMIT 1").collect()[0][0])
except:
    # Fallback - get from widgets or environment
    try:
        primary_account = dbutils.widgets.get("primary_account", "")
        all_accounts_str = dbutils.widgets.get("all_accounts", "[]")
        all_accounts = json.loads(all_accounts_str)
    except:
        logger.error("Could not retrieve configuration from previous notebook - using defaults")
        primary_account = ""
        all_accounts = []

credential = DefaultAzureCredential()

print(f"Primary Purview Account: {primary_account}")
print(f"Total Accounts to Extract: {len(all_accounts)}")


In [ ]:
# SECTION 1: Setup & Load Configuration from Previous Notebook

import sys
import os
from pathlib import Path
import json
import pandas as pd
from datetime import datetime
from typing import List, Dict, Any
import logging

sys.path.insert(0, str(Path.cwd().parent / "src"))

# Load dependencies
from config_manager import get_config_manager
from purview_client import PurviewClient
from azure.identity import DefaultAzureCredential

logger = logging.getLogger(__name__)

# Load config
try:
    config_mgr = get_config_manager()
except:
    logger.warning("Config file not found - using environment variables")

print("✓ Purview extraction notebook initialized")

# Purview Data Governance Extract

Complete extraction of all Purview metadata including:
- Collections and hierarchy
- Data Sources and configurations
- Assets and lineage
- Scans and scan runs
- Classifications and glossary terms
- Integration runtimes
- Business metadata

**Output**: Raw JSON and standardized Spark DataFrames, tagged with source Purview account

**Previous notebook**: 00-Setup.ipynb (validates environment, discovers Purview instances)
